<a href="https://colab.research.google.com/github/aryan30-tp/Deep-learning/blob/main/NLP_using_bow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
keras.utils.set_random_seed(42)

In [2]:
train_url = "https://www.dropbox.com/scl/fi/ito6bnl2yaf1uw0uqibzf/lyric_genre_train.csv?rlkey=04dkn5un2djza8x0bdmfnlw3u&st=y47qh8i4&dl=1"
val_url = "https://www.dropbox.com/scl/fi/xmywjzqsaa8n5sn1bs0t9/lyric_genre_val.csv?rlkey=hggbeo0s1iaxjpa6z80429xl9&st=6i7d8eau&dl=1"
test_url = "https://www.dropbox.com/scl/fi/fnocl69w9ojs9s5zb0xvf/lyric_genre_test.csv?rlkey=z4hjopw7vaihoh948cbb5mvdp&st=xwond7dp&dl=1"

train_df = pd.read_csv(train_url,index_col=0)
val_df = pd.read_csv(val_url,index_col=0)
test_df = pd.read_csv(test_url,index_col=0)

In [3]:
y_train=pd.get_dummies(train_df['Genre']).to_numpy()
y_val=pd.get_dummies(val_df['Genre']).to_numpy()
y_test=pd.get_dummies(test_df['Genre']).to_numpy()

In [4]:
text_vectorization=keras.layers.TextVectorization(
    ngrams=2,
    max_tokens=10000, # Reduced max_tokens to prevent OOM errors
    output_mode='multi_hot'
)

In [5]:
text_vectorization.adapt(train_df['Lyric'])

In [6]:
X_train=text_vectorization(train_df['Lyric'])
X_val=text_vectorization(val_df['Lyric'])
X_test=text_vectorization(test_df['Lyric'])
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of X_test: {X_test.shape}")

Shape of X_train: (48991, 10000)
Shape of X_val: (16331, 10000)
Shape of X_test: (21774, 10000)


In [15]:
input=keras.layers.Input(shape=(10000,))
x=keras.layers.Dense(8,activation='relu')(input)
x=keras.layers.Dropout(0.5)(x)
output=keras.layers.Dense(3,activation='softmax')(x)
model=keras.Model(input,output)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │        80,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,035 (312.64 KB)

 Trainable params: 80,035 (312.64 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [17]:
history=model.fit(X_train, y_train, epochs=4, batch_size=32, validation_data=(X_val, y_val))

Epoch 1/4
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.6545 - loss: 0.7647 - val_accuracy: 0.7353 - val_loss: 0.6007
Epoch 2/4
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7044 - loss: 0.6683 - val_accuracy: 0.7470 - val_loss: 0.5870
Epoch 3/4
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7263 - loss: 0.6245 - val_accuracy: 0.7494 - val_loss: 0.5870
Epoch 4/4
1531/1531 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7378 - loss: 0.6004 - val_accuracy: 0.7505 - val_loss: 0.5927


In [18]:
model.evaluate(x=X_test,y=y_test)

681/681 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7392 - loss: 0.6063


[0.6063156723976135, 0.7392302751541138]

In [20]:
def lyric_predict(p):
  vect_data=text_vectorization([p])
  pred=model.predict(vect_data)
  pred
  print(f"{float(pred[0,0]*100):.2f}% hiphop")
  print(f"{float(pred[0,1]*100):.2f}% pop")
  print(f"{float(pred[0,2]*100):.2f}% rock")

In [27]:
p="""
Legends never die
When the world is calling you
Can you hear them screaming out your name?
Legends never die
They become a part of you
Every time you bleed for reaching greatness
Relentless you survive

They never lose hope when everything's cold and the fighting's near
It's deep in their bones, they'll run into smoke when the fire is fierce
Oh pick yourself up, 'cause

Legends never die
When the world is calling you
Can you hear them screaming out your name?
Legends never die
They become a part of you
Every time you bleed for reaching greatness
Legends never die

They're written down in eternity
But you'll never see the price it costs
The scars collected all their lives

When everything's lost, they pick up their hearts and avenge defeat
Before it all starts, they suffer through harm just to touch a dream
Oh pick yourself up, 'cause

Legends never die
When the world is calling you
Can you hear them screaming out your name?
Legends never die
They become a part of you
Every time you bleed for reaching greatness
Legends never die

When the world is calling out your name
Begging you to fight
Pick yourself up once more
Pick yourself up, 'cause

Legends never die
When the world is calling you
Can you hear them screaming out your name?
Legends never die
They become a part of you
Every time you bleed for reaching greatness
Legends never die
"""
lyric_predict(p)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
0.01% hiphop
8.29% pop
91.70% rock
